# Wavelet-Space Deterministic CNN

This notebook extends the deterministic ERA5 downscaling experiment from
pixel space to an explicit multiscale wavelet representation.

The original 0.25° ERA5 daily maximum 2-m air temperature field is
artificially coarsened by a factor of four. Instead of predicting the
missing residual directly on the spatial grid, the residual is decomposed
using a two-level Daubechies-2 discrete wavelet transform.

The model predicts residual wavelet coefficients at two spatial scales and
reconstructs the fine-resolution temperature field using the inverse
wavelet transform.

## Multiscale representation

A two-level 2-D wavelet decomposition separates the field into:

- Level-2 approximation: A2
- Level-2 horizontal, vertical, and diagonal details: H2, V2, D2
- Level-1 horizontal, vertical, and diagonal details: H1, V1, D1

For a 48 × 60 target field:

- Level-2 coefficients have shape 12 × 15
- Level-1 detail coefficients have shape 24 × 30

The goal is to test whether explicitly organizing temperature variability
by spatial scale improves deterministic reconstruction relative to the
pixel-space residual CNN.

In [ ]:
# ============================================================
# WAVELET-SPACE DETERMINISTIC CNN
# ERA5 TMAX CONTROLLED DOWNSCALING
# ============================================================

from pathlib import Path
import random

import numpy as np
import pandas as pd
import xarray as xr
import pywt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device("cpu")

print("Device:", DEVICE)


# ============================================================
# TRAINING SETTINGS
# ============================================================

BATCH_SIZE = 8
N_EPOCHS = 5
LEARNING_RATE = 1e-3

N_TRAIN = 8000
N_VAL = 1500


# ============================================================
# WAVELET SETTINGS
# ============================================================

WAVELET_NAME = "db2"
WAVELET_LEVEL = 2
WAVELET_MODE = "periodization"

print("Wavelet:", WAVELET_NAME)
print("Level:", WAVELET_LEVEL)
print("Mode:", WAVELET_MODE)
print("PyWavelets version:", pywt.__version__)

## Data

The normalized ERA5 data are not distributed with this repository.

The notebook expects the following files in a local `data/` directory:

- `ERA5_Tmax_train_normalized.nc`
- `ERA5_Tmax_validation_normalized.nc`
- `ERA5_Tmax_test_normalized.nc`

The chronological split is:

- Training: 1950–2005
- Validation: 2006–2014
- Testing: 2015–2024

The original 48 × 60 target field is reduced to 12 × 15 by area averaging
and then bilinearly interpolated back to 48 × 60. The difference between
the ERA5 target and this interpolated field defines the residual to be learned.

In [ ]:
# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path("..")

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)


TRAIN_FILE = DATA_DIR / "ERA5_Tmax_train_normalized.nc"
VAL_FILE = DATA_DIR / "ERA5_Tmax_validation_normalized.nc"
TEST_FILE = DATA_DIR / "ERA5_Tmax_test_normalized.nc"

BEST_MODEL_FILE = CHECKPOINT_DIR / "wavelet_cnn_best.pt"
LATEST_MODEL_FILE = CHECKPOINT_DIR / "wavelet_cnn_latest.pt"

TRAIN_LOSS_FILE = RESULTS_DIR / "wavelet_cnn_train_losses.npy"
VAL_LOSS_FILE = RESULTS_DIR / "wavelet_cnn_val_losses.npy"

print("Project directory:", PROJECT_DIR.resolve())

In [ ]:
# ============================================================
# LOAD NORMALIZED ERA5 DATA
# ============================================================

train_ds = xr.open_dataset(TRAIN_FILE)
val_ds = xr.open_dataset(VAL_FILE)
test_ds = xr.open_dataset(TEST_FILE)

train_da = train_ds["tmax"]
val_da = val_ds["tmax"]
test_da = test_ds["tmax"]

print("Training shape:", train_da.shape)
print("Validation shape:", val_da.shape)
print("Testing shape:", test_da.shape)

In [ ]:
# ============================================================
# BASE CONTROLLED-DOWNSCALING DATASET
# ============================================================

class ERA5DownscalingDataset(Dataset):
    """
    Construct paired ERA5 target and bilinear conditioning fields.

    48 x 60 target
        ->
    12 x 15 coarse field
        ->
    48 x 60 bilinearly interpolated conditioning field
    """

    def __init__(
        self,
        data_array,
        coarse_factor=4
    ):
        self.data = data_array
        self.coarse_factor = coarse_factor


    def __len__(self):
        return self.data.sizes["valid_time"]


    def __getitem__(self, idx):

        field = (
            self.data
            .isel(valid_time=idx)
            .values
            .astype(np.float32)
        )

        target = (
            torch
            .from_numpy(field)
            .unsqueeze(0)
        )

        x = target.unsqueeze(0)


        # ----------------------------------------------------
        # 48 x 60 -> 12 x 15
        # ----------------------------------------------------

        coarse_small = F.interpolate(
            x,
            scale_factor=1 / self.coarse_factor,
            mode="area"
        )


        # ----------------------------------------------------
        # 12 x 15 -> 48 x 60
        # ----------------------------------------------------

        bilinear = F.interpolate(
            coarse_small,
            size=target.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        bilinear = bilinear.squeeze(0)


        return {
            "target": target,
            "bilinear": bilinear
        }

In [ ]:
# ============================================================
# WAVELET ENCODER
# ============================================================

def wavelet_encode(field):
    """
    Encode one 48 x 60 field into two wavelet tensors.

    Returns
    -------
    level2 : np.ndarray
        Shape [4, 12, 15]
        Channels = A2, H2, V2, D2

    level1 : np.ndarray
        Shape [3, 24, 30]
        Channels = H1, V1, D1
    """

    coeffs = pywt.wavedec2(
        field,
        wavelet=WAVELET_NAME,
        level=WAVELET_LEVEL,
        mode=WAVELET_MODE
    )

    cA2 = coeffs[0]

    cH2, cV2, cD2 = coeffs[1]
    cH1, cV1, cD1 = coeffs[2]


    level2 = np.stack(
        [
            cA2,
            cH2,
            cV2,
            cD2
        ],
        axis=0
    ).astype(np.float32)


    level1 = np.stack(
        [
            cH1,
            cV1,
            cD1
        ],
        axis=0
    ).astype(np.float32)


    return level2, level1


# ============================================================
# WAVELET DECODER
# ============================================================

def wavelet_decode(
    level2,
    level1,
    output_shape=(48, 60)
):
    """
    Reconstruct a spatial field from the two wavelet levels.
    """

    coeffs = [
        level2[0],

        (
            level2[1],
            level2[2],
            level2[3]
        ),

        (
            level1[0],
            level1[1],
            level1[2]
        )
    ]


    reconstructed = pywt.waverec2(
        coeffs,
        wavelet=WAVELET_NAME,
        mode=WAVELET_MODE
    )


    # Crop if required after inverse transform
    reconstructed = reconstructed[
        :output_shape[0],
        :output_shape[1]
    ]


    return reconstructed.astype(np.float32)

In [ ]:
# ============================================================
# VERIFY LOSSLESS WAVELET RECONSTRUCTION
# ============================================================

example_dataset = ERA5DownscalingDataset(test_da)

sample = example_dataset[0]

target = (
    sample["target"]
    .squeeze()
    .numpy()
)

bilinear = (
    sample["bilinear"]
    .squeeze()
    .numpy()
)

residual = target - bilinear


level2, level1 = wavelet_encode(
    residual
)


reconstructed_residual = wavelet_decode(
    level2,
    level1,
    output_shape=residual.shape
)


reconstruction_rmse = np.sqrt(
    np.mean(
        (
            reconstructed_residual
            - residual
        ) ** 2
    )
)


print("Residual shape:", residual.shape)

print("Level-2 shape:", level2.shape)
print("Level-1 shape:", level1.shape)

print(
    "Wavelet reconstruction RMSE:",
    reconstruction_rmse
)

print(
    "Original number of values:",
    residual.size
)

print(
    "Wavelet coefficients:",
    level2.size + level1.size
)

In [ ]:
# ============================================================
# WAVELET DATASET
# ============================================================

class WaveletResidualDataset(Dataset):
    """
    Convert target/bilinear pairs into wavelet representations.

    Input:
        bilinear temperature field

    Target:
        wavelet coefficients of
        target - bilinear
    """

    def __init__(self, base_dataset):

        self.base_dataset = base_dataset


    def __len__(self):

        return len(self.base_dataset)


    def __getitem__(self, idx):

        sample = self.base_dataset[idx]


        target = (
            sample["target"]
            .squeeze()
            .numpy()
            .astype(np.float32)
        )


        bilinear = (
            sample["bilinear"]
            .squeeze()
            .numpy()
            .astype(np.float32)
        )


        residual = (
            target
            - bilinear
        )


        # ----------------------------------------------------
        # Conditioning field in wavelet space
        # ----------------------------------------------------

        coarse_L2, coarse_L1 = wavelet_encode(
            bilinear
        )


        # ----------------------------------------------------
        # Target residual in wavelet space
        # ----------------------------------------------------

        residual_L2, residual_L1 = wavelet_encode(
            residual
        )


        return {

            "coarse_L2":
                torch.from_numpy(
                    coarse_L2
                ),

            "coarse_L1":
                torch.from_numpy(
                    coarse_L1
                ),

            "residual_L2":
                torch.from_numpy(
                    residual_L2
                ),

            "residual_L1":
                torch.from_numpy(
                    residual_L1
                ),

            "target":
                torch.from_numpy(
                    target
                ).unsqueeze(0),

            "bilinear":
                torch.from_numpy(
                    bilinear
                ).unsqueeze(0)
        }

In [ ]:
# ============================================================
# CREATE BASE DATASETS
# ============================================================

full_train_base = ERA5DownscalingDataset(
    train_da
)

full_val_base = ERA5DownscalingDataset(
    val_da
)

full_test_base = ERA5DownscalingDataset(
    test_da
)


# ============================================================
# WRAP IN WAVELET DATASET
# ============================================================

full_train_wavelet = WaveletResidualDataset(
    full_train_base
)

full_val_wavelet = WaveletResidualDataset(
    full_val_base
)

test_wavelet = WaveletResidualDataset(
    full_test_base
)


# ============================================================
# SAME CPU-FRIENDLY SUBSETS
# ============================================================

rng = np.random.default_rng(SEED)

train_indices = rng.choice(
    len(full_train_wavelet),
    size=N_TRAIN,
    replace=False
)

val_indices = rng.choice(
    len(full_val_wavelet),
    size=N_VAL,
    replace=False
)


train_dataset = Subset(
    full_train_wavelet,
    train_indices.tolist()
)

val_dataset = Subset(
    full_val_wavelet,
    val_indices.tolist()
)


# ============================================================
# DATA LOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


test_loader = DataLoader(
    test_wavelet,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_wavelet))

In [ ]:
# ============================================================
# CHECK BATCH SHAPES
# ============================================================

batch = next(iter(train_loader))

print(
    "Coarse Level 2:",
    batch["coarse_L2"].shape
)

print(
    "Coarse Level 1:",
    batch["coarse_L1"].shape
)

print(
    "Residual Level 2:",
    batch["residual_L2"].shape
)

print(
    "Residual Level 1:",
    batch["residual_L1"].shape
)

## Wavelet CNN architecture

The model uses a hierarchical two-branch architecture.

The level-2 branch processes the four coarse-scale wavelet channels
(A2, H2, V2, D2) and predicts the corresponding level-2 residual
coefficients.

The learned level-2 features are then bilinearly upsampled to the level-1
resolution and concatenated with the three level-1 conditioning bands
(H1, V1, D1).

This allows information learned at the broader spatial scale to inform the
prediction of finer-scale wavelet coefficients.

In [ ]:
# ============================================================
# WAVELET RESIDUAL CNN
# ============================================================

class WaveletResidualCNN(nn.Module):

    def __init__(self):

        super().__init__()


        # ====================================================
        # LEVEL-2 BRANCH
        #
        # Input:
        # A2, H2, V2, D2
        # Shape: [B, 4, 12, 15]
        # ====================================================

        self.level2_encoder = nn.Sequential(

            nn.Conv2d(
                4,
                24,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                24,
                24,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                24,
                24,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU()
        )


        self.level2_output = nn.Conv2d(
            24,
            4,
            kernel_size=3,
            padding=1
        )


        # ====================================================
        # LEVEL-1 BRANCH
        #
        # Input:
        # 3 level-1 conditioning bands
        # +
        # 24 learned level-2 features
        #
        # Total = 27 channels
        # ====================================================

        self.level1_encoder = nn.Sequential(

            nn.Conv2d(
                27,
                24,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                24,
                24,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                24,
                16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU()
        )


        self.level1_output = nn.Conv2d(
            16,
            3,
            kernel_size=3,
            padding=1
        )


    def forward(
        self,
        coarse_L2,
        coarse_L1
    ):

        # ----------------------------------------------------
        # Level 2
        # ----------------------------------------------------

        features_L2 = self.level2_encoder(
            coarse_L2
        )

        predicted_L2 = self.level2_output(
            features_L2
        )


        # ----------------------------------------------------
        # Transfer learned coarse-scale features to level 1
        # ----------------------------------------------------

        features_L2_up = F.interpolate(
            features_L2,
            size=coarse_L1.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


        # ----------------------------------------------------
        # Combine level-1 coefficients + learned L2 features
        # ----------------------------------------------------

        combined_L1 = torch.cat(
            [
                coarse_L1,
                features_L2_up
            ],
            dim=1
        )


        features_L1 = self.level1_encoder(
            combined_L1
        )


        predicted_L1 = self.level1_output(
            features_L1
        )


        return (
            predicted_L2,
            predicted_L1
        )

In [ ]:
# ============================================================
# INITIALIZE MODEL
# ============================================================

wavelet_model = WaveletResidualCNN().to(
    DEVICE
)


n_params = sum(
    p.numel()
    for p in wavelet_model.parameters()
    if p.requires_grad
)


print(
    f"Trainable parameters: {n_params:,}"
)


# ============================================================
# LOSS AND OPTIMIZER
# ============================================================

criterion = nn.MSELoss()


optimizer = torch.optim.Adam(
    wavelet_model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# TRAIN WAVELET CNN
# ============================================================

wavelet_train_losses = []
wavelet_val_losses = []

best_val_loss = np.inf


for epoch in range(1, N_EPOCHS + 1):

    # ========================================================
    # TRAINING
    # ========================================================

    wavelet_model.train()

    running_loss = 0.0
    n_samples = 0


    for batch in train_loader:

        coarse_L2 = (
            batch["coarse_L2"]
            .to(DEVICE)
        )

        coarse_L1 = (
            batch["coarse_L1"]
            .to(DEVICE)
        )

        target_L2 = (
            batch["residual_L2"]
            .to(DEVICE)
        )

        target_L1 = (
            batch["residual_L1"]
            .to(DEVICE)
        )


        pred_L2, pred_L1 = wavelet_model(
            coarse_L2,
            coarse_L1
        )


        # ----------------------------------------------------
        # Loss at both scales
        # ----------------------------------------------------

        loss_L2 = criterion(
            pred_L2,
            target_L2
        )

        loss_L1 = criterion(
            pred_L1,
            target_L1
        )


        loss = (
            loss_L2
            + loss_L1
        )


        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        batch_size = coarse_L2.shape[0]

        running_loss += (
            loss.item()
            * batch_size
        )

        n_samples += batch_size


    train_loss = (
        running_loss
        / n_samples
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    wavelet_model.eval()

    running_loss = 0.0
    n_samples = 0


    with torch.no_grad():

        for batch in val_loader:

            coarse_L2 = (
                batch["coarse_L2"]
                .to(DEVICE)
            )

            coarse_L1 = (
                batch["coarse_L1"]
                .to(DEVICE)
            )

            target_L2 = (
                batch["residual_L2"]
                .to(DEVICE)
            )

            target_L1 = (
                batch["residual_L1"]
                .to(DEVICE)
            )


            pred_L2, pred_L1 = wavelet_model(
                coarse_L2,
                coarse_L1
            )


            loss_L2 = criterion(
                pred_L2,
                target_L2
            )

            loss_L1 = criterion(
                pred_L1,
                target_L1
            )


            loss = (
                loss_L2
                + loss_L1
            )


            batch_size = coarse_L2.shape[0]

            running_loss += (
                loss.item()
                * batch_size
            )

            n_samples += batch_size


    val_loss = (
        running_loss
        / n_samples
    )


    wavelet_train_losses.append(
        train_loss
    )

    wavelet_val_losses.append(
        val_loss
    )


    # ========================================================
    # SAVE LATEST MODEL
    # ========================================================

    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                wavelet_model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "train_loss":
                train_loss,

            "val_loss":
                val_loss,

            "wavelet":
                WAVELET_NAME,

            "level":
                WAVELET_LEVEL
        },
        LATEST_MODEL_FILE
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    wavelet_model.state_dict(),

                "val_loss":
                    val_loss,

                "wavelet":
                    WAVELET_NAME,

                "level":
                    WAVELET_LEVEL
            },
            BEST_MODEL_FILE
        )

        best_marker = " <-- BEST"

    else:

        best_marker = ""


    print(
        f"Epoch {epoch:02d}/{N_EPOCHS} | "
        f"Train: {train_loss:.8f} | "
        f"Validation: {val_loss:.8f}"
        f"{best_marker}"
    )


# ============================================================
# SAVE LOSSES
# ============================================================

np.save(
    TRAIN_LOSS_FILE,
    np.asarray(
        wavelet_train_losses
    )
)

np.save(
    VAL_LOSS_FILE,
    np.asarray(
        wavelet_val_losses
    )
)


print(
    "\nTraining complete."
)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Best checkpoint:",
    BEST_MODEL_FILE
)

In [ ]:
# ============================================================
# TRAINING CURVE
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 5)
)


epochs = np.arange(
    1,
    len(wavelet_train_losses) + 1
)


ax.plot(
    epochs,
    wavelet_train_losses,
    marker="o",
    label="Training"
)


ax.plot(
    epochs,
    wavelet_val_losses,
    marker="o",
    label="Validation"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Wavelet-space MSE loss"
)

ax.set_title(
    "Wavelet CNN Training"
)

ax.legend()


fig.tight_layout()


figure_file = (
    FIGURES_DIR
    / "wavelet_cnn_training_loss.png"
)


fig.savefig(
    figure_file,
    dpi=200,
    bbox_inches="tight"
)


plt.close(fig)


print(
    "Saved:",
    figure_file
)

In [ ]:
# ============================================================
# LOAD BEST WAVELET CNN
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_FILE,
    map_location=DEVICE
)


wavelet_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


wavelet_model.eval()


print(
    "Loaded best model from epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation loss:",
    checkpoint["val_loss"]
)



# ============================================================
# WAVELET CNN PREDICTION FUNCTION
# ============================================================

def predict_wavelet_cnn(
    model,
    bilinear
):
    """
    Predict a high-resolution field from one
    48 x 60 bilinear conditioning field.
    """

    coarse_L2, coarse_L1 = wavelet_encode(
        bilinear
    )


    coarse_L2_tensor = (
        torch
        .from_numpy(coarse_L2)
        .float()
        .unsqueeze(0)
        .to(DEVICE)
    )


    coarse_L1_tensor = (
        torch
        .from_numpy(coarse_L1)
        .float()
        .unsqueeze(0)
        .to(DEVICE)
    )


    with torch.no_grad():

        predicted_L2, predicted_L1 = (
            model(
                coarse_L2_tensor,
                coarse_L1_tensor
            )
        )


    predicted_L2 = (
        predicted_L2
        .squeeze(0)
        .cpu()
        .numpy()
    )


    predicted_L1 = (
        predicted_L1
        .squeeze(0)
        .cpu()
        .numpy()
    )


    predicted_residual = wavelet_decode(
        predicted_L2,
        predicted_L1,
        output_shape=bilinear.shape
    )


    prediction = (
        bilinear
        + predicted_residual
    )


    return prediction.astype(
        np.float32
    )

In [ ]:
# ============================================================
# FULL HELD-OUT TEST EVALUATION
# 2015–2024
# ============================================================

bilinear_squared_error = 0.0
wavelet_squared_error = 0.0

n_values = 0


for idx in range(
    len(full_test_base)
):

    sample = full_test_base[idx]


    truth = (
        sample["target"]
        .squeeze()
        .numpy()
    )


    bilinear = (
        sample["bilinear"]
        .squeeze()
        .numpy()
    )


    prediction = predict_wavelet_cnn(
        wavelet_model,
        bilinear
    )


    bilinear_squared_error += (
        (
            bilinear
            - truth
        ) ** 2
    ).sum()


    wavelet_squared_error += (
        (
            prediction
            - truth
        ) ** 2
    ).sum()


    n_values += truth.size


bilinear_rmse = np.sqrt(
    bilinear_squared_error
    / n_values
)


wavelet_cnn_rmse = np.sqrt(
    wavelet_squared_error
    / n_values
)


improvement = (
    (
        bilinear_rmse
        - wavelet_cnn_rmse
    )
    /
    bilinear_rmse
    *
    100
)


print(
    "=========================================="
)

print(
    "FULL TEST PERIOD: 2015–2024"
)

print(
    "=========================================="
)


print(
    f"Bilinear RMSE:   "
    f"{bilinear_rmse:.5f}"
)

print(
    f"Wavelet CNN RMSE:"
    f" {wavelet_cnn_rmse:.5f}"
)

print(
    f"Improvement over bilinear: "
    f"{improvement:.2f}%"
)

In [ ]:
# ============================================================
# SAVE BASIC TEST RESULTS
# ============================================================

test_results = pd.DataFrame(
    {
        "method": [
            "Bilinear interpolation",
            "Wavelet CNN"
        ],

        "rmse": [
            bilinear_rmse,
            wavelet_cnn_rmse
        ]
    }
)


results_file = (
    RESULTS_DIR
    / "wavelet_cnn_test_results.csv"
)


test_results.to_csv(
    results_file,
    index=False
)


test_results

## Interpretation

The Wavelet CNN predicts the missing residual in a representation that
explicitly separates spatial scales and orientations.

The full comparison with the Pixel CNN and both conditional flow-matching
models is performed separately in `06_model_evaluation.ipynb`, where all
methods are evaluated on the same 200 held-out test days using RMSE,
spatial variability, gradient error, extreme-temperature metrics, and
spatial power spectra.

### Important limitation

The Pixel CNN and Wavelet CNN are not parameter-matched. Therefore,
improvements observed for the Wavelet CNN should be interpreted as evidence
that the multiscale architecture is promising rather than as proof that the
wavelet representation alone causes the improvement.